In [ ]:
# Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved.
#
# SPDX-License-Identifier: BSD-3-Clause
#
# Portions of this file consist of AI-generated content. AI-assisted
# content has been reviewed and validated by the authors.

# Eyes on the NPU: YOLOv26s through the VitisAI execution provider

The robot's eyes are two YOLOv26s models: *pose* (where you are, as 17 body
keypoints) and *detect* (what to grab, as 80 COCO classes). Both run on the
XDNA2 NPU of your Ryzen AI APU, at camera rate, without touching the CPU
or the iGPU.

This notebook teaches one of the most important skills of the workshop:
**putting an ONNX model on the NPU**. That means building an ONNX Runtime
`InferenceSession` through the VitisAI execution provider, with the compile
cache wired correctly, and then verifying you actually got the NPU.

You will also meet the two classic Ryzen AI failure modes: the silent CPU
fallback and the NPU context budget. Everything generic (letterboxing,
output decoding, overlay drawing) is provided in `common.imaging`. It would
look identical in any YOLO project.

**Matching TODOs:** `1.1`-`1.3` in `project/models/npu.py`.

In [ ]:
# workshop bootstrap: run against the finished reference code in solution/
# (no TODO stubs), so every cell works out of the box. To exercise YOUR
# implementations instead, change "solution" to "project" and restart.
import sys, pathlib
import logging
from pathlib import Path
import shutil
import time

PROJECT = (pathlib.Path.cwd() / ".." / ".." / "solution").resolve()
assert (
    PROJECT / "config" / "workshop.yaml"
).exists(), f"solution not found at {PROJECT}"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from common.quiet import hush_benign_warnings

hush_benign_warnings()  # silence known-benign third-party log noise

from common.config import load_config, resolve, PROJECT_ROOT
import cv2
import onnxruntime as ort
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from jupyter_ui_poll import ui_events
from common.fixtures import tiny_onnx_path
from common.imaging import letterbox, decode_pose, draw_skeleton
from common.imaging import decode_detections, draw_detections
import subprocess

MONITOR = (pathlib.Path.cwd() / ".." / ".." / "launch_monitor.sh").resolve()
subprocess.Popen(
    ["bash", str(MONITOR)],
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

cfg = load_config()
print("running against:", PROJECT_ROOT)

In [ ]:
# Camera-to-notebook plumbing, used by every live cell below.


def to_jpeg(img):
    """BGR frame -> JPEG bytes for a persistent Image widget."""
    ok, jpg = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 80])
    return jpg.tobytes()

## 1. The stack in one picture

![Alt Text](../../animations/ONNX_to_NPU_Stack.gif)

Three facts drive the whole design:

1. **The EP ships with the Ryzen AI wheel, not with stock onnxruntime.** A
   plain `pip install onnxruntime` cannot see the NPU. The rig's venv was
   installed from Ryzen AI SW 1.7.1.
2. **NPU compilation is slow, loading is fast.** The first compile of a model
   takes tens of minutes and is done ONCE (from the Ryzen AI SDK venv, per
   the repo README). The result lands in `cache_dir` under `cache_key`, and
   every later session build loads it in about a second. That is why the
   workshop config points at the repo's existing `cache/`: nothing you do
   today recompiles anything, because we already compiled the models for you
   by running each model once.
3. **Every model needs its own `cache_key`.** Two models sharing a key
   clobber each other's compiled artifacts.

In [ ]:
providers = ort.get_available_providers()
print("providers in this onnxruntime build:", providers)

HAVE_NPU = "VitisAIExecutionProvider" in providers
print("NPU available here:", HAVE_NPU)

On the rig you should see `VitisAIExecutionProvider` in that list. On a
machine without the Ryzen AI stack you'll see only CPU.

## 2. The session factory

Here is the pattern, in full. Read it carefully: this is TODO 1.2, and the
rest of the pipeline stands on it.

In [ ]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("notebook")


def build_session(
    onnx_path,
    device="npu",
    *,
    vitisai_config="config/vaiep_config.json",
    cache_dir="../../cache",
    cache_key="model",
):
    """Create an InferenceSession on the NPU (VitisAI EP) or the CPU."""
    onnx_path = resolve(onnx_path)
    if not onnx_path.exists():
        raise FileNotFoundError(f"{onnx_path} not found")

    # (1) Requesting the NPU on a machine that can't serve it must NOT
    #     crash. Fall back to CPU with a warning: the pipeline runs
    #     (slowly) on any dev box, and the rig logs tell you the truth.
    if (
        device == "npu"
        and "VitisAIExecutionProvider" not in ort.get_available_providers()
    ):
        logger.warning("VitisAI EP not available - falling back to CPU")
        device = "cpu"

    if device == "cpu":
        return ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])

    # (2) The NPU path: ONE provider, configured through provider_options.
    cache = resolve(cache_dir)
    cache.mkdir(parents=True, exist_ok=True)
    return ort.InferenceSession(
        str(onnx_path),
        providers=["VitisAIExecutionProvider"],
        provider_options=[
            {
                "config_file": str(resolve(vitisai_config)),  # compiler passes
                "cache_dir": str(cache),  # compiled artifacts live here
                "cache_key": cache_key,  # UNIQUE per model!
                "target": "VAIML",  # matches the config file
            }
        ],
    )


print("session factory defined")

What each `provider_options` field does:

- **`config_file`** is `config/vaiep_config.json`, the VitisAI compiler
  configuration. Ours selects the `vaiml_partition` pass with
  `enable_f32_to_bf16_conversion: true`, meaning the FP32 ONNX graph is
  partitioned and run in BF16 on the NPU. You export models in plain FP32
  and let the compiler handle the rest.
- **`cache_dir` / `cache_key`** say where the compiled NPU binary lives.
  A cache hit loads in about a second. A cache miss means a compile, which
  is only possible in the SDK venv; the deployment runtime will tell you
  *"Model compilation is not supported in a deployment only installation"*.
- **`target: "VAIML"`** must match the target named in the config file.

## 3. Exercise it offline, on a 107-byte model

Before touching real models, prove the factory works using the tiny embedded
ONNX model from the fixtures (`y = 2x`). Run this cell and test the factory
via the project's self-test harness.

In [ ]:
tiny = tiny_onnx_path(PROJECT_ROOT / "_artifacts")
sess = build_session(tiny, device="cpu", cache_key="nb_tiny")

x = np.arange(4, dtype=np.float32).reshape(1, 4)
(result,) = sess.run(None, {"x": x})
print("x =", x, "\ny =", result, "  (expect 2x)")

## 4. Lesson one: the silent CPU fallback

A session that cannot use the NPU does not error. It runs on the CPU,
10-30x slower. The toy model has no compiled cache, and a deployment
install cannot compile one, so this cell requests the CPU on purpose.
Ask the session what it actually got:

In [ ]:
def active_provider(session):
    # get_providers() lists the session's EPs in priority order;
    # the FIRST entry is what's actually executing your graph.
    return session.get_providers()[0]


# The toy model has no compiled NPU cache, and a deployment-only install
# cannot compile one. Request the CPU and print the provider the session
# actually got.
shutil.rmtree(
    resolve(cfg["system"]["cache_dir"]) / "nb_tiny", ignore_errors=True
)  # a failed compile leaves a partial cache
# dir behind - clear it

request = "cpu"
npu_attempt = build_session(tiny, device=request, cache_key="nb_tiny")
print("Got provider:", active_provider(npu_attempt))

Here that prints `CPUExecutionProvider`, because this cell requested the
CPU. The real pose model below, whose cache entry ships with the repo, is
the one that must print `VitisAIExecutionProvider`. If it ever doesn't,
you want to know.

Make this a reflex: build session, then print the provider. The project's
`report_placement` helper does it for you, and the self-test checks it.

## 5. The real thing: YOLOv26s-pose

Now the actual eyes. The model files live in the repo `models/` directory;
we compiled them earlier. The workshop config points at them, plus the
shared compile cache.

The pose model's export is end-to-end: its output is already
`[1, 300, 57]` = box (4) + score (1) + class (1) + 17 x (x, y, visibility).
No NMS needed. Just threshold and un-letterbox, which `common.imaging`
does for you.


In [ ]:
y = cfg["yolo_pose"]
pose_onnx = resolve(y["onnx"])
pose_sess = None
if pose_onnx.exists():
    t0 = time.perf_counter()
    pose_sess = build_session(
        pose_onnx,
        device=y["device"],
        vitisai_config=y["vitisai_config"],
        cache_dir=cfg["system"]["cache_dir"],
        cache_key=y["cache_key"],
    )
    print(
        f"session built in {time.perf_counter()-t0:.1f}s "
        f"(cache hit is ~1s; a cold compile is 30-60 min in the SDK venv)"
    )
    print("Using provider:", active_provider(pose_sess))
else:
    print(
        f"({pose_onnx} not found - skipping the real-model cells; "
        "from workshops/vvla-pipeline run ./bootstrap.sh, or python scripts/export_yolo26s_pose.py)"
    )

## 6. Watch it live

Time to see yourself. The next cell streams the mount camera through the
pose model and updates the picture in place: stand up, wave, watch the
skeleton follow you. The stream runs in a while loop until you press
**stop**, then you move on to the next cell.


In [ ]:
# Live preview: pose inference on every frame, drawn in place. The loop
# runs until you press the stop button above the image.


def open_mount_camera():
    """The real mount camera: the configured V4L device, else the default webcam."""
    import os

    device = cfg["cameras"]["mount"].get("device", 0)
    candidates = [device, 0]
    if isinstance(device, str) and not pathlib.Path(device).exists():
        candidates = [0]  # configured path absent: go straight to
        # the default webcam (probing a missing
        # path makes OpenCV warn noisily)
    # The AMD ISP node can still be busy for a moment after the previous
    # notebook's kernel releases it. Retry before declaring it stuck.
    for _ in range(8):
        for candidate in candidates:
            cap = (
                cv2.VideoCapture(candidate, cv2.CAP_V4L2)  # V4L2 direct: no
                if isinstance(candidate, str)  # CV_IMAGES probing
                else cv2.VideoCapture(candidate)
            )
            if cap.isOpened():
                print(f"camera open: {candidate}")
                return cap
            cap.release()
        time.sleep(0.25)
    holders = []
    if isinstance(device, str):
        for pid in os.listdir("/proc"):
            if not pid.isdigit():
                continue
            fd_dir = f"/proc/{pid}/fd"
            try:
                fds = os.listdir(fd_dir)
            except OSError:
                continue
            for fd in fds:
                try:
                    target = os.readlink(f"{fd_dir}/{fd}")
                except OSError:
                    continue
                if target != device:
                    continue
                try:
                    cmd = (
                        open(f"/proc/{pid}/cmdline", "rb")
                        .read()
                        .replace(b"\0", b" ")
                        .decode()
                        .strip()
                    )
                except OSError:
                    cmd = ""
                holders.append(f"pid {pid} ({cmd[:140]})")
                break
    held = ("Still held by:\n  " + "\n  ".join(holders) + "\n") if holders else ""
    raise RuntimeError(
        "could not open the mount camera.\n"
        f"  config expects {device}\n"
        f"{held}"
        "Close the other notebook. Its kernel shuts down on close and "
        "releases the camera. Re-run this cell. Restarting "
        "./run_notebooks.sh is not required."
    )


def pose_overlay(img):
    """One frame through the pose model; returns the drawn overlay."""
    if pose_sess is None:
        return img  # no model here: show the raw frame
    nchw, info = letterbox(img, int(y["imgsz"]))
    outputs = pose_sess.run(None, {pose_sess.get_inputs()[0].name: nchw})
    boxes, scores, kpts = decode_pose(
        outputs, info, img.shape, float(y["conf_threshold"])
    )
    return draw_skeleton(img, boxes, scores, kpts, float(y["kpt_threshold"]))


import gc
import signal
import threading


class ReleaseCameraOnInterrupt:
    """Close /dev/video0 when the notebook tab's stop button fires.

    That button sends SIGINT. The pose call can sit inside the NPU driver
    (drm_syncobj_array_wait) and never return to Python, so a finally
    block on the main thread never runs and the camera stays locked.
    sigwait on this side thread receives the interrupt anyway and closes it.
    """

    def __init__(self, cap, stopped):
        self.cap = cap
        self.stopped = stopped
        self.done = False
        self.thread = None

    def __enter__(self):
        signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGINT})
        self.thread = threading.Thread(target=self._watch, daemon=True)
        self.thread.start()
        return self

    def _watch(self):
        while not self.done:
            signal.sigwait([signal.SIGINT])
            if self.done:
                return
            self.stopped["flag"] = True
            try:
                self.cap.release()
            except Exception:
                pass

    def __exit__(self, exc_type, exc, tb):
        self.done = True
        self.stopped["flag"] = True
        if self.thread is not None and self.thread.is_alive():
            signal.pthread_kill(self.thread.ident, signal.SIGINT)
            self.thread.join(timeout=1)
        try:
            self.cap.release()
        except Exception:
            pass
        gc.collect()
        signal.pthread_sigmask(signal.SIG_UNBLOCK, {signal.SIGINT})
        return False


cap = open_mount_camera()
stopped = {"flag": False}
with ReleaseCameraOnInterrupt(cap, stopped):
    ok, first = cap.read()
    if not ok:
        raise RuntimeError("camera opened but returned no frame - re-run this cell")

    stop_btn = widgets.Button(description="stop", button_style="danger")
    stop_btn.on_click(lambda _: stopped.update(flag=True))
    # ONE persistent Image widget, updated in place. Re-displaying a fresh
    # image every frame tears the old DOM node down first - that is what
    # made the view blink. Swapping the bytes on the same widget is
    # flicker-free.
    view = widgets.Image(format="jpeg", width=first.shape[1])
    view.value = to_jpeg(pose_overlay(first.copy()))
    display(widgets.VBox([stop_btn, view]))

    n, t0 = 1, time.time()
    with ui_events() as poll:
        while not stopped["flag"]:
            poll(10)  # let the stop click arrive mid-loop
            ok, frm = cap.read()
            if not ok:
                print("camera stopped mid-stream")
                break
            view.value = to_jpeg(pose_overlay(frm.copy()))
            n += 1
    dt = max(time.time() - t0, 1e-6)
    ran_on = (
        active_provider(pose_sess) if pose_sess is not None else "no model (raw camera)"
    )
    print(f"stopped: {n} frames in {dt:.1f} s = {n/dt:.1f} fps on {ran_on}")
cap = None
gc.collect()

On the NPU expect double-digit millisecond inference and a smooth stream;
on CPU expect hundreds of milliseconds per frame and a slideshow. Same code,
same model, only the provider differs. That is why you always print it.

## 7. Lesson two: the NPU context budget

The XDNA2 partition holds **2 concurrent hardware contexts**. If a pipeline
wants four resident ONNX models at once, that is two too many, and the
symptom you get is:

```
HW context creation unsuccessful ... sub buffer size and offset ...
```

Rules to live by:

- This is **not a stale cache**. A model that works alone and fails at
  startup fails because *others already hold the contexts*.
- Both YOLO models fit together and exactly fill the budget (that's the
  workshop configuration). A latency-tolerant model is moved to the CPU
  deliberately; vision stays on the NPU because it runs at camera rate,
  where latency actually hurts.
- Each model has an explicit `device:` in the config. Because your factory
  falls back gracefully, you can also leave things on `npu`, read the
  startup logs to see what fit, then pin the fields deliberately.

Let's prove the two YOLOs coexist. The next cell loads the detect model
*alongside* the pose session that's still alive in this kernel and streams
the camera through **both models at once** - two live views, side by side:
pose on the left, detect on the right. Wave at the camera while you hold
an object up, watch both overlays track simultaneously, then press
**stop**:


In [ ]:
yd = cfg["yolo_detect"]
det_onnx = resolve(yd["onnx"])

if not det_onnx.exists():
    print(f"({det_onnx} not found - skipping. On the rig, this cell proves the")
    print(" two YOLO models share the NPU: two sessions, two cache keys, one budget.)")
else:
    det_sess = build_session(
        det_onnx,
        device=yd["device"],
        vitisai_config=yd["vitisai_config"],
        cache_dir=cfg["system"]["cache_dir"],
        cache_key=yd["cache_key"],
    )  # its OWN cache key!
    print(
        "pose session   ->",
        active_provider(pose_sess) if pose_sess is not None else "(not built)",
    )
    print("detect session ->", active_provider(det_sess))

    def detect_overlay(img):
        """One frame through the detect model; returns (overlay, detections)."""
        nchw, info = letterbox(img, int(yd["imgsz"]))
        dets = decode_detections(
            det_sess.run(None, {det_sess.get_inputs()[0].name: nchw}),
            info,
            img.shape,
            float(yd["conf_threshold"]),
        )
        return draw_detections(img, dets), dets

    # Both YOLOs resident on the NPU at once - the full context budget in
    # use. ONE camera, and every frame goes through BOTH sessions; the two
    # persistent widgets play side by side.
    cap = open_mount_camera()
    stopped = {"flag": False}
    with ReleaseCameraOnInterrupt(cap, stopped):
        stop_btn = widgets.Button(description="stop", button_style="danger")
        stop_btn.on_click(lambda _: stopped.update(flag=True))
        pose_view = widgets.Image(format="jpeg", width=470)
        det_view = widgets.Image(format="jpeg", width=470)
        label = widgets.Label("starting...")
        display(
            widgets.VBox(
                [
                    stop_btn,
                    widgets.HBox(
                        [
                            widgets.VBox([widgets.Label("YOLOv26s-pose"), pose_view]),
                            widgets.VBox([widgets.Label("YOLOv26s-detect"), det_view]),
                        ]
                    ),
                    label,
                ]
            )
        )

        n, t0 = 0, time.time()
        with ui_events() as poll:
            while not stopped["flag"]:
                poll(10)  # let the stop click arrive mid-loop
                ok, frm = cap.read()
                if not ok:
                    print("camera stopped mid-stream")
                    break
                pose_view.value = to_jpeg(pose_overlay(frm.copy()))
                vis, dets = detect_overlay(frm.copy())
                det_view.value = to_jpeg(vis)
                label.value = (
                    ", ".join(f"{d.name} {d.score:.2f}" for d in dets[:6])
                    or "(nothing above threshold)"
                )
                n += 1
        dt = max(time.time() - t0, 1e-6)
        print(
            f"stopped: {n} frames in {dt:.1f} s = {n/dt:.1f} fps through BOTH models "
            f"on {active_provider(det_sess)} - two sessions, two cache keys, one "
                "context budget, exactly filled"
        )
    cap = None
    gc.collect()

## 8. Your turn

Open **`project/models/npu.py`** and implement:

| TODO | What | The cell that taught it |
|---|---|---|
| **1.1** | `npu_available()` | section 1 (provider list) |
| **1.2** | `build_npu_session(...)` | sections 2-3 (the factory: fallback, CPU path, VitisAI provider_options) |
| **1.3** | `active_provider(session)` | section 4 (the receipt) |

Then, from the `project/` directory:

```bash
python -m models.selftest

# to see all the TODOs you have left
python app.py
```

The harness first checks everything offline (tiny model, mocked VitisAI
wiring: it records the `provider_options` you pass and names any missing
key), then, on the rig, builds the real pose model and saves an overlay
artifact to `_artifacts/pose_overlay.png`.

Next notebook: `02_cpu_mediapipe.ipynb`, the fingers.